In [1]:
from __future__ import annotations
import os
import re
import bw2data
from bw2data import get_activity
from dataclasses import dataclass
from typing import Dict, Iterable, Literal, Optional, Tuple
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
#activate the bw project
#bw2data.projects.set_current("ei311")

In [4]:
import sys
sys.path.append('../../utils/') 
from dpLCIA import * 

## input file needed: 
- dynLCI_file_path: dynamic LCI 
- ghg_dir:  all FaIR metrics, with uncertainty, we calc here, not via BW2 

In [5]:
hydro_files_2030 = {
    ("SSP1-19", 2030): "../dp-LCI_output/hydro_dpLCI_v2_reservoir/market_for_electricity_hydro_high_voltage_CA-QC_SSP1-VLLO_2030.xlsx",
    ("SSP5-85", 2030): "../dp-LCI_output/hydro_dpLCI_v2_reservoir/market_for_electricity_hydro_high_voltage_CA-QC_SSP5-H_2030.xlsx",
}

hydro_files_2050 = {
    ("SSP1-19", 2050): "../dp-LCI_output/hydro_dpLCI_v2_reservoir/dateshifted/market_for_electricity_hydro_high_voltage_CA-QC_SSP1-VLLO_2050.xlsx",
    ("SSP5-85", 2050): "../dp-LCI_output/hydro_dpLCI_v2_reservoir/dateshifted/market_for_electricity_hydro_high_voltage_CA-QC_SSP5-H_2050.xlsx",
}

ghg_dir = "../../../dpLCIA/FaIR_dpCFs/output/metrics"   # directory containing all CO2/CH4/N2O Excel files

# RF is the radiative forcing metric to use
metric = "rf"   # or "agwp"  

In [6]:
results_rf = compute_dynamic_lcia_for_scenarios(
    elec_files=hydro_files_2030,
    ghg_dir=ghg_dir,
    metric="rf",    
)

print(f"Scenarios computed: {list(results_rf.keys())}")

Scenarios computed: [('SSP1-19', 2030), ('SSP5-85', 2030)]


In [7]:
results_agwp = compute_dynamic_lcia_for_scenarios(
    elec_files=hydro_files_2030,
    ghg_dir=ghg_dir,
    metric="agwp",  
)

print(f"Scenarios computed: {list(results_agwp.keys())}")

Scenarios computed: [('SSP1-19', 2030), ('SSP5-85', 2030)]


#### now compute 2050 LCIA, make sure corrected hydro 2050 path

In [8]:
results_rf_2050 = compute_dynamic_lcia_for_scenarios(
    elec_files=hydro_files_2050,
    ghg_dir=ghg_dir,
    metric="rf",    
)
#print(f"Scenarios computed: {list(results_rf.keys())}") # wrong print out 

results_agwp_2050 = compute_dynamic_lcia_for_scenarios(
    elec_files=hydro_files_2050,
    ghg_dir=ghg_dir,
    metric="agwp",  
)
#print(f"Scenarios computed: {list(results_agwp.keys())}")

Scenarios computed: [('SSP1-19', 2030), ('SSP5-85', 2030)]
Scenarios computed: [('SSP1-19', 2030), ('SSP5-85', 2030)]


In [9]:
### saving the results as pickle
from pathlib import Path
import pickle

out_dir =  Path.cwd() / "results_updated_CO2CH4"
out_dir.mkdir(exist_ok=True)

# 1) Pickles (full objects, preserves DatetimeIndex/DataFrames)
rf_pkl   = out_dir / "results_rf_2030.pkl"
agwp_pkl = out_dir / "results_agwp_2030.pkl"

rf_pkl_2050   = out_dir / "results_rf_2050.pkl"
agwp_pkl_2050 = out_dir / "results_agwp_2050.pkl"


# save 2030 
with open(rf_pkl, "wb") as f:
    pickle.dump(results_rf, f, protocol=pickle.HIGHEST_PROTOCOL)

with open(agwp_pkl, "wb") as f:
    pickle.dump(results_agwp, f, protocol=pickle.HIGHEST_PROTOCOL)



### save 2050 
with open(rf_pkl_2050, "wb") as f:
    pickle.dump(results_rf_2050, f, protocol=pickle.HIGHEST_PROTOCOL)

with open(agwp_pkl_2050, "wb") as f:
    pickle.dump(results_agwp_2050, f, protocol=pickle.HIGHEST_PROTOCOL)



##### BW2 mapping

#####  start dpLCIA

##### compute dynamic LCIA per subtype (central + 5–95 % band)